# 02 - Event Study

Reads the event-study output (`data/processed/event_study.parquet`) and ranks events by composite impact score and by event type. Falls back to `data/sample/event_study_sample.parquet` on a fresh clone.

**Composite impact score:** `|cum_log_return(t, t+7)| + max(realized_vol_post / realized_vol_pre - 1, 0)`. It captures both directional move and volatility expansion in a single non-negative number.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
candidates = [ROOT / 'data' / 'processed' / 'event_study.parquet', ROOT / 'data' / 'sample' / 'event_study_sample.parquet']
es_path = next(p for p in candidates if p.exists())
es = pd.read_parquet(es_path)
print(f'Loaded {es_path.relative_to(ROOT)}')
print(f'Shape: {es.shape}')
es.head(3)

## Mean impact by event type

In [ ]:
by_type = (es.groupby('event_type')['impact_score']
             .agg(['count', 'mean', 'median'])
             .sort_values('mean', ascending=False)
             .round(3))
by_type

In [ ]:
ax = by_type['mean'].plot(kind='barh', figsize=(8, 4),
                          title='Mean composite impact score by event type', color='#5B8FF9')
ax.invert_yaxis(); ax.set_xlabel('impact score'); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

Protocol Upgrade and Exchange events are associated with the largest mean short-window reactions in the sample. ETF and CPI events show lower average impact, consistent with substantial pre-event pricing - large positioning typically takes place before scheduled or well-anticipated catalysts.

## Top 10 individual events by impact

In [ ]:
top = (es.sort_values('impact_score', ascending=False)
          .head(10)
          [['event_date', 'event_type', 'event_name', 'asset',
            'ret_t_t7', 'vol_ratio', 'impact_score']]
          .round(3))
top

## Event-type by asset heatmap

In [ ]:
pivot = es.pivot_table(index='event_type', columns='asset', values='impact_score', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_title('Mean impact score by event-type x asset')
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## Interpretation

These numbers are descriptive short-window market reactions associated with each event category. They are **not** strict causal estimates - other contemporaneous information may have moved markets in the same window. The ranking is robust because the impact score is non-negative and the formula does not weight by curated severity, so subjectivity in the severity rubric does not drive the ordering.